In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pickle
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv("./cyberbullying_data_labelled.csv")

In [ ]:

print("=" * 60)
print("✅ USING YOUR LOADED DATASET")
print("=" * 60)
print(f"📊 Total samples: {len(df)}")

# Use your cleaned_text column (better than raw Text)
text_column = 'cleaned_text'
label_column = 'oh_label'

X = df[text_column].astype(str)
y = df[label_column].astype(int)

print(f"✅ Text column: '{text_column}'")
print(f"✅ Label column: '{label_column}'")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📚 Training: {len(X_train)} samples")
print(f"🧪 Testing: {len(X_test)} samples")


In [ ]:
# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=10000,     # More features for better accuracy
    ngram_range=(1, 2),     # Words + word pairs
    stop_words='english',
    min_df=2,
    max_df=0.9
)

print("\n⏳ Training model... (this may take 30-60 seconds)")
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train model
model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
model.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)

print("=" * 60)
print("🎯 MODEL TRAINING COMPLETE!")
print("=" * 60)
print(f"✅ Accuracy: {accuracy:.2%}")
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Bullying', 'Bullying']))

In [ ]:
def check_comment(comment):
    """
    Check if a comment is cyberbullying
    """
    # Clean input (same as your cleaned_text)
    processed = str(comment).lower().strip()
    
    # Transform and predict
    comment_tfidf = vectorizer.transform([processed])
    prediction = model.predict(comment_tfidf)[0]
    probabilities = model.predict_proba(comment_tfidf)[0]
    
    bullying_prob = probabilities[1] * 100
    safe_prob = probabilities[0] * 100
    
    # Determine result
    if prediction == 1:
        result = "🚨 BULLYING DETECTED"
        risk = "HIGH" if bullying_prob > 80 else "MODERATE" if bullying_prob > 60 else "LOW"
        emoji = "🔴"
    else:
        result = "✅ NOT BULLYING"
        risk = "SAFE"
        emoji = "🟢"
    
    # Display
    print("\n" + "=" * 60)
    print(f"{emoji} CYBERBULLYING DETECTION RESULT")
    print("=" * 60)
    print(f"📝 Comment: \"{comment}\"")
    print(f"\n🔍 Result: {result}")
    print(f"📊 Confidence: {max(bullying_prob, safe_prob):.1f}%")
    print(f"⚖️ Risk Level: {risk}")
    print(f"\n📈 Probabilities:")
    print(f"   🔴 Bullying: {bullying_prob:.2f}%")
    print(f"   🟢 Safe: {safe_prob:.2f}%")
    print("=" * 60)
    
    return prediction



In [ ]:
print("\n" + "=" * 60)
print("🧪 TESTING WITH SAMPLE COMMENTS")
print("=" * 60)

# Test with some examples
test_comments = [
    "You're so ugly and stupid, nobody likes you",
    "Go kill yourself loser",
    "Thanks for helping me with homework",
    "I think you made a good point in class",
    "Fat pig, stop eating so much",
    "Everyone hates you, just disappear",
    "Can we meet at the library tomorrow?",
    "You're worthless and should die",
    "I love your new haircut!",
    "Why are you such a failure at everything"
]

for i, comment in enumerate(test_comments, 1):
    print(f"\nTest #{i}:")
    check_comment(comment)

In [ ]:
print("\n" + "=" * 60)
print("🔍 INTERACTIVE MODE")
print("Paste any comment to analyze")
print("Type 'quit' to exit")
print("=" * 60)

while True:
    try:
        user_input = input("\n>>> Paste your comment: ")
        
        if user_input.lower().strip() in ['quit', 'exit', 'q']:
            print("\n👋 Goodbye! Stay safe online!")
            break
        
        if user_input.strip():
            check_comment(user_input)
        else:
            print("⚠️ Please enter a comment.")
            
    except KeyboardInterrupt:
        print("\n\n👋 Exiting...")
        break


In [ ]:
pickle.dump(model, open('cyberbullying_model.pkl', 'wb'))
pickle.dump(vectorizer, open('vectorizer.pkl', 'wb'))

print("\n" + "=" * 60)
print("💾 MODEL SAVED!")
print("=" * 60)
print("Files saved in your notebook folder:")
print("  • cyberbullying_model.pkl")
print("  • vectorizer.pkl")